In [6]:
from pathlib import Path
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt


#loading initial imputs and outputs 
X= np.load('../data/initial_data/function_4/initial_inputs.npy')
y = np.load('../data/initial_data/function_4/initial_outputs.npy')

#checking shape of data 
print('Inputs Shape:', X.shape)
print('Outputs Shape:', y.shape)

#creating a dataframe 
df=pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4'])
df['y']=y

# --- add new observation for Function 3 ---
new_row = pd.DataFrame([{
    'x1': 0.01,
    'x2': 0.01,
    'x3': 0.99,
    'x4': 0.99,
    'y': -39.0288607
}, {'x1': 0.43, 'x2': 0.36, 'x3': -0.29, 'x4': 0.43, 'y': -0.764565297}, {"x1": 0.50,
    "x2": 0.50,
    "x3": 0.01,
    "x4": 0.36,
    "y":  -9.846862892}, {
        'x1': 0.99,
        'x2': 0.01,
        'x3': 0.01,
        'x4': 0.01,
        'y': -31.97607733
    }, {
        'x1': 0.36,
        'x2': 0.36,
        'x3': 0.50,
        'x4': 0.43,
        'y': -0.92388271
    }, {
        'x1': 0.50,
        'x2': 0.29,
        'x3': 0.43,
        'x4': 0.43,
        'y': -1.935139502
    }
])

# append to df
df = pd.concat([df, new_row], ignore_index=True)

# rebuild X and y from the *updated* df
X = df[['x1', 'x2', 'x3', 'x4']].to_numpy()
y = df['y'].to_numpy()

# sanity check
print("len(df):", len(df))
print("Inputs Shape:", X.shape)
print("Outputs Shape:", y.shape)
print(df.tail())

# Check
#checking DataFrame
df.head(50)



Inputs Shape: (30, 4)
Outputs Shape: (30,)
len(df): 36
Inputs Shape: (36, 4)
Outputs Shape: (36,)
      x1    x2    x3    x4          y
31  0.43  0.36 -0.29  0.43  -0.764565
32  0.50  0.50  0.01  0.36  -9.846863
33  0.99  0.01  0.01  0.01 -31.976077
34  0.36  0.36  0.50  0.43  -0.923883
35  0.50  0.29  0.43  0.43  -1.935140


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


In [8]:
#setting parameters and making  grid 
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


#hyperparamters for GP surrogate model and acqusition function 
rbf_lengthscale = 0.5
noise_assumption = 0.3
beta = 2.5

eps = 0.01    # small padding away from the edges
g = np.linspace(eps, 1 - eps, 15)

x1g, x2g, x3g, x4g = np.meshgrid(g, g, g, g)

X_grid = np.column_stack([x1g.ravel(), x2g.ravel(), x3g.ravel(), x4g.ravel()])


x1g, x2g, x3g, x4g = np.meshgrid(g, g, g, g, indexing='ij')
X_grid = np.column_stack([x1g.ravel(), x2g.ravel(), x3g.ravel(), x4g.ravel()])

print(X_grid.shape)


(50625, 4)


In [11]:
#fit the gp to current data
kernel = (C(1.0, (1e-3, 1e3)) * Matern(length_scale=0.3, length_scale_bounds=(1e-4, 5), nu=2.5) 
    + WhiteKernel(noise_level=0.3, noise_level_bounds=(1e-10, 1)))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=0)
gp.fit(X, y)

GaussianProcessRegressor(kernel=1**2 * Matern(length_scale=0.3, nu=2.5) + WhiteKernel(noise_level=0.3),
                         n_restarts_optimizer=10, normalize_y=True,
                         random_state=0)

In [13]:
#compute acquisition UCB on the grid 
#posterior mean and std 
mu, std = gp.predict(X_grid, return_std = True)
#std = np.maximum(std, 1e-12)

#UCB acquistion 
ucb= mu + beta * std

In [15]:
#choose next query point 

ix = int(np.argmax(ucb))
x_next = X_grid[ix]
print(f"x_next = ({x_next[0]:.6f}, {x_next[1]:.6f},{x_next[2]:.6f},{x_next[3]:.6f}) ")

x_next = (0.430000, 0.430000,0.430000,0.430000) 


In [17]:
#x_next check 
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances

y_best = float(np.max(y))
model        = gp
X_candidates = np.array(X_grid)   # shape (n_candidates, n_dims)
acq_values   = np.array(ucb)   # shape (n_candidates,)
x_next       = np.array(x_next)   # shape (n_dims,)
y_best       = y_best
X_sampled    = np.array(X)   # all evaluated X so far


def _percentile_of_score(values, score):
    """Simple percentile rank: percentage of values <= score."""
    values = np.asarray(values).ravel()
    return float(100.0 * (np.sum(values <= score) / max(len(values), 1)))

def _first_length_scale_from_kernel(kernel):
    """Extract effective GP length-scale (handles RBF, Matern, etc.)."""
    if hasattr(kernel, "length_scale"):
        ls = getattr(kernel, "length_scale")
        try:
            ls = float(np.mean(ls))
        except Exception:
            ls = float(ls)
        return ls
    for attr in ("k1", "k2"):
        if hasattr(kernel, attr):
            ls = _first_length_scale_from_kernel(getattr(kernel, attr))
            if ls is not None:
                return ls
    return None

def _nearest_dist_over_ell(x_next, X_sampled, ell):
    """Compute distance from x_next to nearest sampled point ÷ ell."""
    x_next = np.asarray(x_next, dtype=float).reshape(1, -1)
    X_sampled = np.asarray(X_sampled, dtype=float)
    if X_sampled.size == 0:
        return np.nan
    dists = euclidean_distances(x_next, X_sampled).ravel()
    d_min = float(np.min(dists))
    if (ell is None) or (ell <= 0):
        return np.nan
    return d_min / ell

def _classify_strategy(sig_pct):
    """Assign strategy label from σ percentile (bounded exploration)."""
    if sig_pct <= 40:
        return "Exploit"
    elif sig_pct <= 75:
        return "Balanced"
    elif sig_pct <= 85:
        return "Explore"
    else:
        return "Too exploratory"

def sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled):
    """
    Builds a 1-row DataFrame with:
      μ_raw (raw), σ_pct (scaled %), dist_over_ℓ_raw (raw), acq_pct (scaled %), Strategy
    """

    # Ensure arrays are the correct shape
    X_candidates = np.asarray(X_candidates, dtype=float)
    acq_values = np.asarray(acq_values, dtype=float).ravel()
    x_next = np.asarray(x_next, dtype=float).ravel()

    # 1) Predict μ, σ for all candidates (for percentiles) and for x_next
    mu_all, sigma_all = model.predict(X_candidates, return_std=True)
    mu_xn, sigma_xn = model.predict(x_next.reshape(1, -1), return_std=True)
    mu_xn = float(mu_xn.ravel()[0])
    sigma_xn = float(sigma_xn.ravel()[0])

    # 2) Percentiles for σ and acquisition
    sig_pct = _percentile_of_score(sigma_all, sigma_xn)

    # Find acquisition value at x_next by matching to nearest candidate
    idx_closest = np.argmin(np.sum((X_candidates - x_next.reshape(1, -1))**2, axis=1))
    acq_xn = float(acq_values[idx_closest])
    acq_pct = _percentile_of_score(acq_values, acq_xn)

    # 3) Distance / ell
    ell = None
    if hasattr(model, "kernel_"):
        ell = _first_length_scale_from_kernel(model.kernel_)
    dist_over_ell = _nearest_dist_over_ell(x_next, X_sampled, ell)

    # 4) Strategy classification
    strategy = _classify_strategy(sig_pct)

    # 5) Build the 1-row DataFrame (rounded for readability)
    data = {
        "μ_raw (raw)":            np.round(mu_xn, 3),
        "σ_pct (scaled %)":       int(np.round(sig_pct)),
        "dist_over_ℓ_raw (raw)":  np.round(dist_over_ell, 3) if np.isfinite(dist_over_ell) else np.nan,
        "acq_pct (scaled %)":     int(np.round(acq_pct)),
        "Strategy":               strategy
    }

    return pd.DataFrame([data])
   

result_df = sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled)
result_df

,μ_raw (raw),σ_pct (scaled %),dist_over_ℓ_raw (raw),acq_pct (scaled %),Strategy
0,-0.966,1,0.099,100,Exploit
